# មេរៀនទី 13 - អង្គចងចាំភ្នាក់ងារជាមួយក្រាហ្វចំណេះដឹង Cognee


## ការតំឡើង

កំណត់ចំណាំនេះបង្ហាញពីរបៀបបង្កើត **ជំនួយការសរសេរកូដឆ្លាត** ដែលមានអនុស្សារណៈរយៈពេលវែង ដោយប្រើ [**Cognee**](https://www.cognee.ai/) ក្រាហ្វចំណេះដឹង និង **Microsoft Agent Framework** (MAF).

Cognee បម្លែងអត្ថបទដែលមិនមានរចនាសម្ព័ន្ធទៅជាក្រាហ្វចំណេះដឹងដែលមានរចនាសម្ព័ន្ធ និងអាចស្វែងរកបាន ដែលគាំទ្រដោយការបញ្ចូលវ៉ិចទ័រ — ផ្តល់ឱ្យភ្នាក់ងាររបស់អ្នកនូវអនុស្សារណៈរយៈពេលវែងដែលមានសមត្ថភាព និងយល់ពីទំនាក់ទំនង។

### អ្វីដែលអ្នកនឹងរៀន
1. **សាងសង់ក្រាហ្វចំណេះដឹង**: បម្លែងប្រវត្តិរូបអ្នកអភិវឌ្ឍន៍ និងវិធីអនុវត្តល្អៗ ទៅជាចំណេះដឹងដែលមានរចនាសម្ព័ន្ធ និងអាចស្វែងរកបាន។
2. **បញ្ចូល Cognee ជាមួយ MAF**: ប្រើមុខងារ `@tool` ដើម្បីឲ្យភ្នាក់ងារ MAF អាចស្វែងរកព័ត៌មានពីក្រាហ្វចំណេះដឹងរបស់ Cognee។
3. **សន្ទនាដែលដឹងពីសេស្យុង**: រក្សាបរិបទខ្សែការសន្ទនារវាងសំណួរច្រើននៅក្នុងសេស្យុងដូចគ្នា។
4. **អនុស្សារណៈរយៈពេលវែង**: រក្សាចំណេះដឹងសំខាន់ៗឲ្យមាននៅឆ្លងកាត់សេស្យុង និងយកវាឡើងវិញក្នុងការសន្ទនាថ្មី។

### តម្រូវការមុន
- Python 3.9+
- Redis កំពុងដំណើរការនៅក្នុងម៉ាស៊ីនក្នុងស្រុក (`docker run -d -p 6379:6379 redis`) សម្រាប់ការគ្រប់គ្រងសេស្យុង
- កូនសោ API សម្រាប់ LLM (ឧ. OpenAI) — កំណត់ `LLM_API_KEY` នៅក្នុង `.env`
- `CACHING=true` នៅក្នុង `.env` (ត្រូវការ​សម្រាប់សេស្យុង Cognee)
- គម្រោង Azure AI Foundry ដែលមានម៉ូដែលជជែកបានដាក់ប្រើ
- `AZURE_AI_PROJECT_ENDPOINT` និង `AZURE_AI_MODEL_DEPLOYMENT_NAME` នៅក្នុង `.env`
- Azure CLI បានចូលគណនី (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## ប្រភេទនៃអង្គចងចាំរបស់ភ្នាក់ងារ

កំណត់ត្រានេះស្វែងយល់ពីប្រភេទអង្គចងចាំបីដដែលពីកំណត់ត្រា Lesson 13 ហើយប៉ុន្តែប្រើ Cognee ជាផ្នែកខាងក្រោយសម្រាប់អង្គចងចាំរយៈពេលវែង:

| ប្រភេទអង្គចងចាំ | មេកានិច | រយៈពេល |
|---|---|---|
| **កំពុងដំណើរការ** | `agent.create_session()` (MAF) | សន្ទនាតែមួយ |
| **រយៈពេលខ្លី** | ឃាសសេស្យុង Cognee (Redis) | សម័យតែមួយ |
| **រយៈពេលវែង** | ក្រាហ្វចំណេះដឹងរបស់ Cognee + វិចទ័រ | ថេរជាយូរអង្វែង |

### ស្ថាបត្យកម្មអង្គចងចាំរបស់ Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## រៀបចំការផ្ទុក Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## ផ្នែក 1 — ការបង្កើតមូលដ្ឋានចំណេះដឹង

យើងបញ្ចូលបីប្រភេទទិន្នន័យដើម្បីបង្កើតមូលដ្ឋានចំណេះដឹងទូលំទូលាយសម្រាប់ជំនួយករកូដរបស់យើង:

1. **ប្រវត្តិអ្នកអភិវឌ្ឍន៍** — ជំនាញផ្ទាល់ខ្លួន និងបរិបទបច្ចេកទេស
2. **អនុវិធីល្អៗសម្រាប់ Python** — ទស្សនៈនៃ Python ជាមួយណែនាំអនុវត្តដែលជាគន្លឹះ
3. **សន្ទនាអតីតកាល** — សម័យសំណួរ និងចម្លើយពីអតីតកាល រវាងអ្នកអភិវឌ្ឍន៍ និងជំនួយការបញ្ញាសិប្បនិម្មិត


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## បង្ហាញក្រាហ្វចំណេះដឹង

Cognee អាចបង្ហាញការមើលឃើញ HTML អន្តរកម្មនៃអង្គធាតុ និងទំនាក់ទំនងដែលវាបានដកយក។


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## ពង្រឹងចងចាំជាមួយ Memify

`memify()` វិភាគក្រាហ្វនៃចំណេះដឹង ហើយបង្កើតកฎឆ្លាតវៃ — កំណត់លំនាំ អនុវត្តិល្អបំផុត និងទំនាក់ទំនងរវាងគំនិត។


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## ផ្នែកទី 2 — ភ្នាក់ងារ MAF ជាមួយឧបករណ៍ Cognee

ឥឡូវនេះ យើងបង្កើតភ្នាក់ងារ MAF ដែលអាចស្នើសុំទៅកាន់ក្រាហ្វចំណេះដឹងរបស់ Cognee តាមរយៈមុខងារ `@tool`។ វាអនុញ្ញាតឱ្យភ្នាក់ងារ ប្រើសមត្ថភាពពេញលេញនៃការស្វែងរកអត្ថន័យដែលមានការយល់ដឹងពីក្រាហ្វ ខណៈដែលរក្សាបរិបទសន្ទនាតាមសម័យ។


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## ចងចាំការងារជាមួយសម័យ

`AgentSession` (បង្កើតតាមរយៈ `agent.create_session()`) ផ្តល់នូវចងចាំសម្រាប់ការងារនៅក្នុងសម័យមួយ។ ភ្នាក់ងារ​អាចយោងត្រឡប់ទៅកាន់សារមុនៗ ខណៈពេលដែលក៏កំពុងស្នើសុំទិន្នន័យពីក្រាហ្វចំណេះដឹងរយៈពេលវែងរបស់ Cognee ផងដែរ។


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## សម័យថ្មី — អង្គចងចាំរយៈពេលវែងនៅតែមាន

ការចាប់ផ្តើមសម័យថ្មីនឹងសម្អាតអង្គចងចាំកំពុងដំណើរការ ប៉ុន្តែក្រាហ្វចំណេះដឹង Cognee នៅតែមាន។ ភ្នាក់ងារអាចយកចំណេះដឹងរយៈពេលវែងដដែលត្រឡប់មកវិញក្នុងសន្ទនាថ្មីមួយទាំងស្រុង។


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Summary

In this notebook you built a coding assistant that combines **MAF's working memory** (`agent.create_session()`) with **Cognee's long-term knowledge graph**.

### What You've Learned
1. **Knowledge graph construction**: Cognee ingests unstructured text and builds a graph + vector memory.
2. **Graph enrichment with memify**: Derived facts and richer relationships on top of your existing graph.
3. **MAF + Cognee integration**: `@tool` functions let an MAF agent query Cognee's graph naturally.
4. **Working memory + long-term memory**: `AgentSession` (via `agent.create_session()`) provides session context while Cognee provides persistent knowledge.
5. **Filtered search with NodeSets**: Target specific subsets of the knowledge graph (e.g. only principles).

### Key Takeaways
- **Cognee** turns raw text into structured, relationship-aware memory — more powerful than a flat vector store.
- **`@tool` functions** bridge MAF agents and external knowledge systems cleanly.
- **`AgentSession`** (via `agent.create_session()`) keeps per-conversation context separate from long-lived knowledge.
- The same knowledge graph serves multiple sessions and agents.

### Real-World Applications
- **Developer copilots**: Code review, incident analysis, architecture assistants
- **Customer-facing copilots**: Support agents over product docs, FAQs, and CRM notes
- **Internal expert copilots**: Policy, legal, or security assistants reasoning over guidelines
- **Unified data layers**: Combine structured and unstructured data into one queryable graph

### Next Steps
- Experiment with temporal awareness in Cognee
- Define an OWL ontology for domain-specific graph quality
- Add user feedback loops to improve retrieval over time
- Scale to multi-agent systems sharing the same Cognee memory layer


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
ឯកសារនេះត្រូវបានបកប្រែដោយប្រើសេវាបកប្រែ AI [Co-op Translator](https://github.com/Azure/co-op-translator). ខណៈពេលដែលយើងខិតខំក្នុងការធានាភាពត្រឹមត្រូវ សូមយកចិត្តទុកដាក់ថាការបកប្រែដោយស្វ័យប្រវត្តិអាចមានកំហុស ឬមិនត្រឹមត្រូវ។ ឯកសារដើមក្នុងភាសាមាតុភូមិគួរត្រូវបានចាត់ទុកជាប្រភពដែលមានសុពលភាព។ សម្រាប់ព័ត៌មានដែលសំខាន់ពិសេស សូមពិចារណាបកប្រែដោយអ្នកជំនាញមនុស្ស។ យើងមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកប្រែខុសណាមួយដែលកើតឡើងពីការប្រើប្រាស់ការបកប្រែនេះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
